# 第七章：循环神经网络 (RNN) — 从序列理解时间

## CNN 理解空间，RNN 理解时间——参数共享从空间维度推广到时间维度

上一章 CNN 通过滑动窗口共享参数检测空间模式（边缘、纹理）。本章把"参数共享"的思想推广到时间维度：同一组权重在序列的每个时间步复用，检测时间模式（趋势、周期、语义依赖）。从 RNN 到 LSTM 再到 Word2Vec，完成从序列建模到自然语言处理的过渡。

## 7.1 从空间到时间：CNN 为什么处理不了序列

### CNN 的两个硬编码假设

CNN 通过两个归纳偏置获得了极高效率：**局部感受野**——每个神经元只看邻近像素，远距离无连接；**参数共享**——同一组权重扫描整张图。这两个假设对图像天然成立，对序列数据全部失效。

考虑句子"猫吃了鱼"。CNN 的 3×3 卷积核在图片上从左到右滑动——覆盖 (猫,吃)、(吃,了)、(了,鱼) 三对相邻词。但"猫"和"鱼"的语义关系（施事-受事）跨了 2 个位置，3×3 核看不到。要让深层神经元覆盖"猫 → 鱼"的依赖，需要堆叠很多层卷积+池化——但序列长度不确定（可能 5 个词也可能 50 个词），无法提前知道需要多少层。

### 序列数据对模型提出的新要求

| 要求 | 为什么 CNN/MLP 做不到 |
|------|----------------------|
| 变长输入 | CNN 要求固定尺寸输入，序列长度不定 |
| 顺序敏感 | 打乱像素顺序，CNN 输出不变；打乱词序，语义全变 |
| 长距离依赖 | "我 30 年前出生在北京……所以我喜欢吃炸酱面"——300 字前的信息需要影响当前词 |

### 解法：时间维度的参数共享

CNN 的参数共享在空间维度——同一核扫描图片的所有位置。RNN 的参数共享在**时间维度**——同一组权重处理序列的每一个时间步。不论序列多长，参数量不变。每一步接收两个输入：当前时刻的数据 + 上一时刻留下的隐藏状态（记忆）。这个循环结构就是 RNN 的核心。

## 7.2 循环神经网络：用隐藏状态传递时间信息

### 核心思想

RNN 在每个时间步做同一件事：读入当前输入 $\mathbf{x}_t$，结合上一时间步的隐藏状态 $\mathbf{h}_{t-1}$，产生新的隐藏状态 $\mathbf{h}_t$ 和当前输出 $\mathbf{y}_t$：

$$\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b})$$

$$\mathbf{y}_t = W_y \mathbf{h}_t + \mathbf{b}_y$$

关键部分是 $W_h \mathbf{h}_{t-1}$——当前隐藏状态不是只依赖当前输入，还依赖上一个隐藏状态，而上一个隐藏状态又依赖上上个，以此类推。展开 $t=3$ 时的隐藏状态：

$$\mathbf{h}_3 = \tanh(W_h \cdot \tanh(W_h \cdot \tanh(W_h \mathbf{h}_0 + W_x \mathbf{x}_1 + \mathbf{b}) + W_x \mathbf{x}_2 + \mathbf{b}) + W_x \mathbf{x}_3 + \mathbf{b})$$

每一步的隐藏状态包含了**当前输入 + 之前所有时间步的压缩信息**。$\mathbf{h}_0$ 通常初始化为全零向量。

### 权重共享：为什么 RNN 只需要很少的参数

所有时间步使用相同的权重矩阵 $W_h$、$W_x$、$W_y$。无论序列长度是 5 还是 500，参数量不变——这是 RNN 能处理任意长度序列的原因。对比 MLP 把整个序列展平为一个大向量 + 大矩阵乘法，RNN 的参数量少得多，但代价是计算必须串行——必须等 $t-1$ 算完才能算 $t$。

### BPTT：随时间反向传播的梯度灾难

RNN 的反向传播沿着时间展开。损失 $\mathcal{L}$ 对早期隐藏状态的梯度需要经过所有中间时间步：

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}_T} \cdot \frac{\partial \mathbf{h}_T}{\partial \mathbf{h}_{T-1}} \cdot \frac{\partial \mathbf{h}_{T-1}}{\partial \mathbf{h}_{T-2}} \cdots \frac{\partial \mathbf{h}_2}{\partial \mathbf{h}_1}$$

每一项 $\frac{\partial \mathbf{h}_k}{\partial \mathbf{h}_{k-1}}$ 包含 $\tanh'(\cdot) \cdot W_h^T$。$\tanh'$ 的值在 $(0, 1]$ 之间。当序列长度 $T$ 增大时，连乘的结果要么指数衰减（$\tanh' < 1$ 主导）要么指数增长（$W_h$ 的谱半径 > 1 主导）——RNN 无法稳定学习超过约 20 个时间步的依赖关系。

这就是 BPTT 的梯度消失/爆炸问题——不是代码 bug，是 RNN 结构本身的数学局限性。


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

# === RNN Cell 从零实现 ===
class RNNCell:
    """单步 RNN: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)"""
    def __init__(self, input_dim, hidden_dim):
        # Xavier 初始化
        self.W_xh = np.random.randn(hidden_dim, input_dim) * np.sqrt(2.0 / (input_dim + hidden_dim))  # 标准正态分布随机数
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * np.sqrt(2.0 / (hidden_dim + hidden_dim))  # 标准正态分布随机数
        self.b_h = np.zeros((hidden_dim, 1))  # 创建全零数组
    
    def forward(self, x_t, h_prev):
        """x_t: (input_dim, 1), h_prev: (hidden_dim, 1)"""
        z = self.W_xh @ x_t + self.W_hh @ h_prev + self.b_h
        return np.tanh(z)  # 双曲正切激活函数

# 演示：滚动序列
cell = RNNCell(input_dim=3, hidden_dim=5)
h = np.zeros((5, 1))  # 创建全零数组

# 输入序列：3 个 token，每个 token 3 维
sequence = [np.array([[1.0], [0.5], [0.0]]),  # 创建 NumPy 数组
            np.array([[0.0], [1.0], [0.5]]),  # 创建 NumPy 数组
            np.array([[0.5], [0.0], [1.0]])]  # 创建 NumPy 数组

print("RNN 前向传播:")
for t, x_t in enumerate(sequence):
    h = cell.forward(x_t, h)
    print(f"  t={t}: h = {h.T[0].round(3)}")
print(f"\n最终隐藏状态记录了前 3 步的信息 (input_dim=3, hidden_dim=5)")


## 7.3 LSTM：用门控机制绕过梯度衰减

### RNN 的根本问题

RNN 的 $\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b})$ 每次更新都**完全重写**隐藏状态——上一个 $\mathbf{h}_{t-1}$ 的内容被 $\tanh$ 压缩后与当前输入混合。经过若干步后，早期信息被反复稀释，逐渐消失。

LSTM 的解决方案：不再重写，而是**选择性更新**。把隐藏状态改为细胞状态 $\mathbf{C}_t$，每次更新时决定"保留多少旧信息、写入多少新信息"。

### 三个门的作用

LSTM 在每一步计算三个门控信号——都是 Sigmoid 输出 $(0, 1)$ 的值，控制信息流动的比例：

**遗忘门 $f_t$**：决定从上一个细胞状态中丢弃多少

$$f_t = \sigma(W_f \cdot [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f)$$

$f_t$ 的每个维度是一个介于 0 和 1 的数。接近 0 表示"忘记这个维度的信息"，接近 1 表示"保留"。

**输入门 $i_t$**：决定往细胞状态中写入多少新信息

$$i_t = \sigma(W_i \cdot [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i)$$

**候选值 $\tilde{\mathbf{C}}_t$**：本次要写入的新信息的内容

$$\tilde{\mathbf{C}}_t = \tanh(W_C \cdot [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_C)$$

### 细胞状态的更新：LSTM 的核心创新

$$\mathbf{C}_t = f_t \odot \mathbf{C}_{t-1} + i_t \odot \tilde{\mathbf{C}}_t$$

第一项 $f_t \odot \mathbf{C}_{t-1}$ 是"选择性遗忘"——逐元素乘遗忘门，保留想要的信息。第二项 $i_t \odot \tilde{\mathbf{C}}_t$ 是"选择性写入"——逐元素乘输入门，加入新信息。

这个更新方式的关键在于：**$\mathbf{C}_t$ 对 $\mathbf{C}_{t-1}$ 的梯度是 $f_t$**——不含 $\tanh'$ 或其他激活函数的导数约束：

$$\frac{\partial \mathbf{C}_t}{\partial \mathbf{C}_{t-1}} = f_t$$

当遗忘门 $f_t \approx 1$ 时，梯度无损回传——即便跨几十个时间步，早期信息仍能影响当前决策。当 $f_t \approx 0$ 时，模型主动选择遗忘。梯度不会被激活函数衰减，只被模型自己的决策（遗忘门）控制——这就是 LSTM 解决梯度消失的根本机制。

### 输出门：从细胞状态到隐藏状态

$$\mathbf{o}_t = \sigma(W_o \cdot [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o)$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{C}_t)$$

输出门决定"细胞状态的哪些部分应该暴露给当前输出"。细胞状态 $\mathbf{C}_t$ 是内部记忆，隐藏状态 $\mathbf{h}_t$ 是外部输出——两者分离是 LSTM 的另一个关键设计，允许模型在内部保持长距离信息的同时，对外输出与当前时间步相关的部分。

### LSTM vs GRU：两种门控的工程选择



GRU (2014, Cho et al.) 是 LSTM (Long Short-Term Memory) 的简化版——性能相近但参数少 25%，训练更快。

### GRU vs LSTM

| 特性 | LSTM | GRU |
|------|------|-----|
| 门数量 | 3 (遗忘/输入/输出) | 2 (重置/更新) |
| 状态数量 | 2 (c, h) | 1 (h) |
| 参数量 | 4× 组 | 3× 组 |
| 长距离依赖 | 最强 | 稍弱但足够 |

#### GRU 更新规则

**重置门 $r_t$**（控制忽略多少过去信息）：

$$

r_t = \sigma(W_r \cdot [h_{t-1}, x_t] + b_r)

$$

**更新门 $z_t$**（控制保留多少过去信息 vs 新信息）：

$$

z_t = \sigma(W_z \cdot [h_{t-1}, x_t] + b_z)

$$

**候选隐藏状态**：

$$

\tilde{h}_t = \tanh(W \cdot [r_t \odot h_{t-1}, x_t] + b)

$$

**最终隐藏状态**（$z_t$ 作为插值系数）：

$$

h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t

$$

关键：当 $z_t \approx 0$，$h_t \approx h_{t-1}$——信息可以无损传递（类似 LSTM 的 carousel）。


### 7.3.1 BPTT 的 3 时间步展开

以 $T=3$、单隐藏层的 RNN 为例，完整写出梯度回传路径。RNN 的前向定义为：

$$\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b})$$

损失 $L = L_1 + L_2 + L_3$（每个时间步的损失之和）。对 $W_h$ 的梯度需要考虑所有时间步的影响：

$$\frac{\partial L}{\partial W_h} = \sum_{t=1}^{3} \frac{\partial L_t}{\partial W_h}$$

展开 $t=3$ 这一项。$L_3$ 通过 $\mathbf{h}_3$ 影响 $W_h$，而 $\mathbf{h}_3$ 又依赖 $\mathbf{h}_2$ 和 $\mathbf{h}_1$：

$$\frac{\partial L_3}{\partial W_h} = \underbrace{\frac{\partial L_3}{\partial \mathbf{h}_3} \cdot \frac{\partial \mathbf{h}_3}{\partial W_h}}_{\text{直接影响}} + \underbrace{\frac{\partial L_3}{\partial \mathbf{h}_3} \cdot \frac{\partial \mathbf{h}_3}{\partial \mathbf{h}_2} \cdot \frac{\partial \mathbf{h}_2}{\partial W_h}}_{\text{通过 }\mathbf{h}_2} + \underbrace{\frac{\partial L_3}{\partial \mathbf{h}_3} \cdot \frac{\partial \mathbf{h}_3}{\partial \mathbf{h}_2} \cdot \frac{\partial \mathbf{h}_2}{\partial \mathbf{h}_1} \cdot \frac{\partial \mathbf{h}_1}{\partial W_h}}_{\text{通过 }\mathbf{h}_1}$$

三项分别对应梯度信号从 $t=3$ 回传到 $W_h$ 的三种路径：直接、跳一步、跳两步。每跳一步都乘以一个雅可比矩阵 $\frac{\partial \mathbf{h}_k}{\partial \mathbf{h}_{k-1}}$。

这个雅可比矩阵的元素来自 $\tanh$ 的导数（$\leq 1$）和 $W_h$ 的谱半径。若 $W_h$ 的最大特征值 $< 1$，连乘后指数衰减为 0——梯度消失。若 $> 1$——梯度爆炸。这就是 BPTT 中梯度不稳定性的数学根源。


In [ ]:
# === LSTM Cell 从零实现 ===
import numpy as np
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))  # 限制数值范围，防止溢出

class LSTMCell:
    def __init__(self, input_dim, hidden_dim):
        dim = input_dim + hidden_dim
        # 四组参数合并（计算效率）
        scale = np.sqrt(2.0 / dim)  # 平方根
        self.W_f = np.random.randn(hidden_dim, dim) * scale  # forget gate
        self.b_f = np.zeros((hidden_dim, 1)) + 1.0  # bias=1 鼓励初始时记住
        
        self.W_i = np.random.randn(hidden_dim, dim) * scale  # input gate
        self.b_i = np.zeros((hidden_dim, 1))  # 创建全零数组
        
        self.W_c = np.random.randn(hidden_dim, dim) * scale  # candidate
        self.b_c = np.zeros((hidden_dim, 1))  # 创建全零数组
        
        self.W_o = np.random.randn(hidden_dim, dim) * scale  # output gate
        self.b_o = np.zeros((hidden_dim, 1))  # 创建全零数组
    
    def forward(self, x_t, h_prev, c_prev):
        combined = np.vstack([h_prev, x_t])  # (hidden+input, 1)
        
        f_t = sigmoid(self.W_f @ combined + self.b_f)    # 遗忘门
        i_t = sigmoid(self.W_i @ combined + self.b_i)    # 输入门
        c_tilde = np.tanh(self.W_c @ combined + self.b_c) # 候选记忆
        o_t = sigmoid(self.W_o @ combined + self.b_o)    # 输出门
        
        c_t = f_t * c_prev + i_t * c_tilde   # 细胞状态更新
        h_t = o_t * np.tanh(c_t)             # 隐藏状态
        
        return h_t, c_t  # 返回结果

# 演示 LSTM
lstm_cell = LSTMCell(input_dim=3, hidden_dim=5)
h = np.zeros((5, 1))  # 创建全零数组
c = np.zeros((5, 1))  # 创建全零数组

print("LSTM 前向传播 (3 个时间步):")
for t, x_t in enumerate(sequence):
    h, c = lstm_cell.forward(x_t, h, c)
    print(f"  t={t}: h={h.T[0].round(3)}")
    print(f"       c={c.T[0].round(3)}")
    print(f"       c 的范数: {np.linalg.norm(c):.2f} (不会指数衰减)")  # 计算向量/矩阵范数


### 7.3.2 遗忘门消除梯度消失的数学推导

LSTM 的状态更新由下式控制：

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

其中 $f_t = \sigma(W_f \cdot [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f)$ 是遗忘门的输出（$0 < f_t < 1$），$i_t \odot \tilde{C}_t$ 是新信息的贡献。

关键是**不需要通过 $\tanh$ 或 Sigmoid 的导数**来计算 $C_t$ 对 $C_{t-1}$ 的梯度：

$$\frac{\partial C_t}{\partial C_{t-1}} = f_t$$

（严格地说是 $\text{diag}(f_t)$，但逐元素分析足够）。这个导数仅由**遗忘门本身的输出值**决定，不再受激活函数的导数值约束。

对比 RNN 的隐藏状态更新 $\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + \cdots)$，其梯度为：

$$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \text{diag}(\tanh'(\cdot)) \cdot W_h^T$$

$\tanh'$ 的值在 $[0, 1]$ 之间，配合 $W_h$ 连乘后快速衰减。而 LSTM 的 $\frac{\partial C_t}{\partial C_{t-1}} = f_t$ 没有激活函数的导数因子——梯度信号通过细胞状态的路径是一个**纯线性**的加法门控，不会指数衰减。

**直观结论**：当遗忘门 $f_t \approx 1$（"记住"），梯度无损耗回传；当 $f_t \approx 0$（"忘记"），梯度被阻断——这正是 LSTM 的设计意图：选择性记忆，而非无差别衰减。


## 7.4 NLP (Natural Language Processing) 基础：从文字到向量

### Tokenization（分词）

深度学习模型不能直接处理文字——需要将文字转换为数字序列。

**三种粒度：**

| 方法 | 示例 | 优点 | 缺点 |
|------|------|------|------|
| 词级 (Word) | "I love NLP" → [1, 42, 78] | 直观 | 词汇表巨大，OOV 问题 |
| 字符级 (Char) | "love" → [l, o, v, e] | 词汇表小 | 序列过长，语义弱 |
| 子词级 (Subword/BPE) | "unbelievable" → [un, believ, able] | **业界标准** | 实现较复杂 |

### Word Embeddings（词嵌入）

将离散的 token ID 映射为连续的稠密向量，使得语义相似的词在向量空间中距离更近：

$$

\mathbf{e}(\text{"king"}) - \mathbf{e}(\text{"man"}) + \mathbf{e}(\text{"woman"}) \approx \mathbf{e}(\text{"queen"})

$$

**Word2Vec (2013, Mikolov)** 开创了词嵌入时代：
- **Skip-gram**：用中心词预测上下文词
- **CBOW**：用上下文词预测中心词

**GloVe (Global Vectors for Word Representation) (2014, Pennington)** 基于全局词共现矩阵的因子分解。

现代 LLM 中，词嵌入是 Transformer 的第一层（可学习的 Embedding 层）。


In [ ]:
# === PyTorch: LSTM 字符级语言模型 ===
# 任务：用前几个字符预测下一个字符（最简单的语言建模）

# 准备数据
import torch
text = "hello world! this is a simple character level language model demo. " * 20
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"词汇表大小: {vocab_size}")
print(f"字符集: {''.join(chars)}")

# 创建训练序列
seq_length = 20
X_data, y_data = [], []
for i in range(0, len(text) - seq_length):
    seq_in = text[i:i + seq_length]
    seq_out = text[i + 1:i + seq_length + 1]
    X_data.append([char_to_idx[c] for c in seq_in])
    y_data.append([char_to_idx[c] for c in seq_out])

X_tensor = torch.tensor(X_data, dtype=torch.long)  # 从数据创建张量
y_tensor = torch.tensor(y_data, dtype=torch.long)  # 从数据创建张量
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"训练样本: {len(X_data)}, 每批: 64")


In [ ]:
# === LSTM 字符级语言模型 ===
import torch
import torch.nn as nn
import torch.optim as optim
class CharLSTM(nn.Module):  # 所有网络层的基类
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)  # 词嵌入层
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers,  # 长短期记忆网络
                            batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim, vocab_size)  # 全连接层 y=Wx+b
    
    def forward(self, x, hidden=None):
        x = self.embedding(x)  # (batch, seq_len, embed_dim)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)  # (batch, seq_len, vocab_size)
        return out, hidden  # 返回结果

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CharLSTM(vocab_size).to(device)  # 将数据移到 GPU 或 CPU
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

print(f"模型参数: {sum(p.numel() for p in model.parameters()):,}")
print(model)


### 7.4.1 Skip-gram 的目标函数

Word2Vec 的 Skip-gram 模型以中心词 $w_t$ 预测上下文词 $w_{t+j}$（$j \in \{-c, \ldots, c\}, j \neq 0$）。给定中心词的 embedding $\mathbf{v}_{w_t}$（输入向量）和上下文词的 embedding $\mathbf{u}_{w_{t+j}}$（输出向量），条件概率定义为 softmax：

$$P(w_{t+j} \mid w_t) = \frac{\exp(\mathbf{u}_{w_{t+j}}^T \mathbf{v}_{w_t})}{\sum_{w \in \mathcal{V}} \exp(\mathbf{u}_w^T \mathbf{v}_{w_t})}$$

分母需要对整个词表 $\mathcal{V}$ 求和——当 $\mathcal{V}$ 包含百万词时不可行。**负采样**将这个 softmax 近似为二分类问题：

$$\mathcal{L} = -\log \sigma(\mathbf{u}_{w_o}^T \mathbf{v}_{w_c}) - \sum_{k=1}^{K} \mathbb{E}_{w_n \sim P_n} \left[\log \sigma(-\mathbf{u}_{w_n}^T \mathbf{v}_{w_c})\right]$$

其中 $\sigma$ 是 sigmoid 函数。第一项鼓励正样本（真实上下文词）的内积大；第二项鼓励 $K$ 个随机采样的噪声词内积小。$K$ 通常取 5–20。

训练完成后，输入向量 $\mathbf{v}$ 和输出向量 $\mathbf{u}$ 都可以作为词嵌入使用，但通常取 $\mathbf{v}$（或二者的平均）。这些向量捕捉了词语的语义相似性——相似用法的词在向量空间中靠近。


In [ ]:
# === 训练 ===
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
history = []
for epoch in range(15):
    model.train()  # 切换到训练模式（启用 Dropout/BatchNorm）
    total_loss = 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)  # 将数据移到 GPU 或 CPU
        
        optimizer.zero_grad()  # 清空上一轮的梯度
        output, _ = model(x_batch)
        loss = criterion(output.reshape(-1, vocab_size), y_batch.reshape(-1))  # 改变张量形状
        loss.backward()  # 反向传播，计算梯度
        optimizer.step()  # 用累积的梯度更新参数
        total_loss += loss.item()  # 取出单个数值
    
    avg_loss = total_loss / len(loader)
    history.append(avg_loss)
    if epoch % 3 == 0:
        print(f"Epoch {epoch:2d}: loss = {avg_loss:.4f}")

# 画训练曲线
plt.figure(figsize=(8, 3))  # 创建画布
plt.plot(history, 'b-')  # 画线图
plt.xlabel('Epoch'); plt.ylabel('Loss')  # 设置x轴标签
plt.title('Char-LSTM Training')  # 设置图表标题
plt.grid(True, alpha=0.3)  # 显示网格线
plt.savefig('../fig/lstm_training.png', dpi=100, bbox_inches='tight')  # 保存图片到文件
plt.show()  # 显示图表

# === 文本生成 ===
@torch.no_grad()
def generate(model, seed_text, length=100, temperature=0.8):
    model.eval()  # 切换到评估模式（禁用 Dropout/BatchNorm）
    chars_gen = list(seed_text)
    hidden = None
    
    input_seq = torch.tensor([[char_to_idx[c] for c in seed_text]], device=device)  # 从数据创建张量
    
    for _ in range(length):
        output, hidden = model(input_seq, hidden)
        # 取最后一个时间步的预测，应用温度
        logits = output[0, -1, :] / temperature
        probs = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, 1).item()  # 取出单个数值
        
        chars_gen.append(idx_to_char[next_idx])
        # 用新字符作为下一步输入
        input_seq = torch.tensor([[next_idx]], device=device)  # 从数据创建张量
    
    return ''.join(chars_gen)  # 返回结果

seed = "hello "
generated = generate(model, seed, length=80, temperature=0.7)
print(f"\n生成文本 (seed: '{seed}'):")
print(generated)


### 7.4.2 序列模型全景对比

| 模型 | 年份 | 核心机制 | 适用场景 | 状态 |
|------|------|---------|---------|------|
| **RNN (Recurrent Neural Network)** | 1986 | 简单循环连接 | 教学、简单序列 | 仅教学用 |
| **LSTM** | 1997 | 门控 + 细胞状态 | 长序列、预测 | 仍在广泛使用 |
| **GRU** | 2014 | 简化门控 | 类似 LSTM 但更轻量 | 仍在广泛使用 |
| **BiLSTM** | - | 双向 LSTM | 序列标注、NER | 工业界常见 |
| **Seq2Seq + Attention** | 2014 | Encoder-Decoder + 注意力 | 机器翻译 | 被 Transformer 取代 |
| **Transformer** | 2017 | 纯注意力、无循环 | **几乎所有 NLP** | 绝对主流 |

### 何时选择 RNN/LSTM 而非 Transformer？

- **数据量少**（< 10K 样本）：LSTM 收敛更快
- **需要流式处理**（逐 token 处理，不需要完整序列）
- **时间序列预测**：LSTM 的归纳偏置（时间连续性）在此任务上仍然有效
- **资源受限**：小 LSTM 比小 Transformer 表现更好


### 7.4.3 NLP (Natural Language Processing) 任务全景

在进入 Transformer/LLM 章节之前，了解 NLP 的经典任务体系：

| 任务 | 输入 → 输出 | 示例 |
|------|-----------|------|
| **文本分类** | 文本 → 标签 | 情感分析、垃圾邮件检测 |
| **序列标注** | token序列 → 标签序列 | NER、词性标注 |
| **文本生成** | prompt → 续写 | 故事生成、代码补全 |
| **机器翻译** | 源语言 → 目标语言 | en→zh |
| **文本摘要** | 长文本 → 短文本 | 新闻摘要 |
| **问答 (QA)** | (问题, 上下文) → 答案 | SQuAD |
| **文本蕴含 (NLI)** | (前提, 假设) → {蕴含,矛盾,中性} | MNLI |

这些任务在 LLM 时代被统一为"文本到文本"的范式——但理解它们各自的难度和评估方法仍然重要。


## 本章小结

| 概念 | 核心公式/机制 |
|------|-------------|
| **RNN (Recurrent Neural Network)** | $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1})$，参数共享 |
| **梯度消失** | $\prod \partial h_k/\partial h_{k-1}$ 连乘指数衰减 |
| **LSTM** | 遗忘门 + 输入门 + 输出门 → 线性细胞状态更新 |
| **GRU** | 重置门 + 更新门（LSTM 简化版） |
| **Tokenization** | 词级/字符级/子词级 (BPE) |
| **Word2Vec** | Skip-gram + CBOW → 语义空间的向量表示 |
| **语言模型** | 给定前文预测下一个 token: $P(x_t | x_{<t})$ |

✅ 掌握了 RNN/LSTM 和 NLP 基础，下一章进入 Transformer 和 LLM 的世界，你会发现一切都有迹可循。
